# **Load model**

In [ ]:
from qwen.models.enc_dec import QwenCrossAttentionEncDec
from transformers import AutoTokenizer
from peft import PeftModel

model = QwenCrossAttentionEncDec.from_pretrained("hai2131/SailorED-pretrain-2epoch")

# my_model = PeftModel.from_pretrained(my_base_model, "hai2131/SailorED-896-sft-contrastive-alignment-lambda-1")

# # merge LoRA vào
# my_model = my_model.merge_and_unload()
tokenizer = AutoTokenizer.from_pretrained("hai2131/SailorED-pretrain-2epoch")

In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

In [ ]:
model_encoder = model.encoder

# **Load dataset**

In [ ]:
!gdown 1ZV80tMSLljhrBMNptcWzExDFw0HEkCxk -O data.zip
!unzip data.zip -d data

In [ ]:
from datasets import load_dataset

vie_sentences = load_dataset(
    "csv",
    data_files="/kaggle/working/data/flores.vi.csv"
)

khm_sentences = load_dataset(
    "csv",
    data_files="/kaggle/working/data/flores.km.csv"
)

lao_sentences = load_dataset(
    "csv",
    data_files="/kaggle/working/data/flores.lo.csv"
)

# **Get embeddings**

In [ ]:
def get_embeddings(sentences, batch_size=8):

    embeddings = []

    for i in range(0, len(sentences), batch_size):

        batch = sentences[i:i+batch_size]

        inputs = tokenizer(
            batch,
            padding=True,
            truncation=True,
            return_tensors="pt"
        ).to(model_encoder.device)

        with torch.no_grad():
            outputs = model_encoder(
                input_ids=inputs["input_ids"],
                attention_mask=inputs["attention_mask"],
                use_cache=False,
                output_hidden_states=True,
                return_dict=True
            )

        hidden_states = outputs.hidden_states
        print(len(hidden_states))

        layer_outputs = []

        for layer in range(0, len(hidden_states), 1):  # layer chẵn

            hidden = hidden_states[layer]

            mask = inputs["attention_mask"].unsqueeze(-1)

            pooled = (hidden * mask).sum(1) / mask.sum(1)

            layer_outputs.append(pooled.cpu())

        embeddings.append(torch.stack(layer_outputs))

    embeddings = torch.cat(embeddings, dim=1)

    return embeddings

# **Visualize**

In [ ]:
import torch
# eng_embeddings = get_embeddings(eng_sentences)
lao_embeddings = get_embeddings(lao_sentences["train"]["sentence"])
vie_embeddings = get_embeddings(vie_sentences["train"]["sentence"])
# tha_embeddings = get_embeddings(tha_sentences)
khm_embeddings = get_embeddings(khm_sentences["train"]["sentence"])

In [ ]:
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE


palette = {
    # "eng":"#1f77b4",
    "lao":"#ff7f0e",
    "vie":"#2ca02c",
    # "tha":"#d62728",
    "khm":"#9467bd"
}

In [ ]:
import torch
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from matplotlib.lines import Line2D

def plot_multilingual_tsne_kde(
    embeddings_dict,
    sample_size=None,
    layers=None,
    perplexity=30,
    cols=5,
    figsize_scale=3,
    random_state=42,
):
    """
    Plot t-SNE KDE distributions for multilingual embeddings across layers
    with legend inside each subplot.
    """

    langs = list(embeddings_dict.keys())

    palette = dict(zip(
        langs,
        sns.color_palette("tab10", len(langs))
    ))

    first_tensor = next(iter(embeddings_dict.values()))
    num_layers = first_tensor.shape[0]

    if layers is None:
        layers = range(45, num_layers)

    if sample_size is None:
        sample_size = first_tensor.shape[1]

    rows = (len(layers) + cols - 1) // cols

    fig, axes = plt.subplots(
        rows,
        cols,
        figsize=(figsize_scale * cols, figsize_scale * rows)
    )

    axes = axes.flatten()

    # tạo legend handles 1 lần
    legend_handles = [
        Line2D([0], [0], color=palette[lang], lw=2, label=lang)
        for lang in langs
    ]

    for idx, layer in enumerate(layers):
        ax = axes[idx]

        X_list = []
        labels = []

        for lang, emb in embeddings_dict.items():
            X_list.append(emb[layer][:sample_size])
            labels.extend([lang] * sample_size)

        X = torch.cat(X_list, dim=0).cpu().numpy()

        tsne = TSNE(
            n_components=2,
            perplexity=perplexity,
            random_state=random_state
        )
        X2 = tsne.fit_transform(X)

        df = pd.DataFrame({
            "x": X2[:, 0],
            "y": X2[:, 1],
            "lang": labels
        })

        for lang in langs:
            subset = df[df.lang == lang]

            sns.kdeplot(
                data=subset,
                x="x",
                y="y",
                levels=6,
                linewidths=2,
                color=palette[lang],
                ax=ax
            )

        ax.set_title(f"Layer {layer}")
        ax.set_xticks([])
        ax.set_yticks([])

        # ✅ legend riêng từng subplot
        ax.legend(
            handles=legend_handles,
            loc="upper right",
            frameon=False,
            fontsize=8
        )

    # hide unused axes
    for idx in range(len(layers), len(axes)):
        axes[idx].axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
plot_multilingual_tsne_kde(
    embeddings_dict={
        "lao": lao_embeddings,
        "vie": vie_embeddings,
        "khm": khm_embeddings,
    },
    sample_size=2009
)